In [2]:
import pandas as pd

# Load the CSV
df = pd.read_csv("url,webpage,result.csv")

# Display structure info
df.info()

# Show column names
print("Columns:", df.columns.tolist())

# Preview the first few rows (optional)
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79987 entries, 0 to 79986
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Record ID  79987 non-null  int64 
 1   URL        79987 non-null  object
 2   Webpage    79987 non-null  object
 3   Result     79987 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 2.4+ MB
Columns: ['Record ID', 'URL', 'Webpage', 'Result']
   Record ID                                                URL  \
0          1                 http://intego3.info/EXEL/index.php   
1          2           https://www.mathopenref.com/segment.html   
2          3   https://www.computerhope.com/issues/ch000254.htm   
3          4  https://www.investopedia.com/terms/n/next-elev...   
4          5                  https://jobs.emss.org.uk/lcc.aspx   

                 Webpage  Result  
0  1613573972338075.html       1  
1  1635698138155948.html       0  
2  1635699228889266.html       0  
3  16357500621

In [3]:
import pandas as pd

# Load the CSV
df = pd.read_csv("url,webpage,result.csv")

# Remove .html or .htm (case-insensitive), then convert to integer
df["Webpage"] = df["Webpage"].str.replace(r"\.html?$", "", case=False, regex=True)
df["Webpage"] = df["Webpage"].astype(int)

# (Optional) Rename to match other files
df.rename(columns={"Webpage": "page_id"}, inplace=True)

# Save cleaned version
df.to_csv("url_webpage_result_cleaned.csv", index=False)
print(" Cleaned CSV saved as 'url_webpage_result_cleaned.csv'")

 Cleaned CSV saved as 'url_webpage_result_cleaned.csv'


In [4]:
df2 = pd.read_csv("url_webpage_result_cleaned.csv")

df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79987 entries, 0 to 79986
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Record ID  79987 non-null  int64 
 1   URL        79987 non-null  object
 2   page_id    79987 non-null  int64 
 3   Result     79987 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 2.4+ MB


In [5]:
import pandas as pd

# Load both datasets
features_df = pd.read_csv("extracted_features.csv")
url_df = pd.read_csv("url_webpage_result_cleaned.csv")

# Convert 'page_id' in both files to integer
features_df["page_id"] = features_df["page_id"].astype(int)
url_df["page_id"] = url_df["page_id"].astype(int)

# Merge on 'page_id'
merged_df = pd.merge(features_df, url_df, on="page_id", how="inner")

# Drop 'URL' column if it exists
if "URL" in merged_df.columns:
    merged_df.drop(columns=["URL"], inplace=True)

# Save the final merged dataset
merged_df.to_csv("final_merged_dataset_clean.csv", index=False)
print(" Final dataset saved as 'final_merged_dataset_clean.csv'")


Final dataset saved as 'final_merged_dataset_clean.csv'


In [6]:
import pandas as pd

# Load your dataset
df = pd.read_csv('final_merged_dataset_clean.csv')

# Drop 'page_id' and 'Record ID' columns if they exist
df = df.drop(columns=['page_id', 'Record ID'], errors='ignore')

# Save the new dataset without these columns
df.to_csv('final_dataset.csv', index=False)

In [1]:
import pandas as pd

# Load your dataset
df = pd.read_csv("final_dataset.csv")

# Rename 'Result' to 'label'
df = df.rename(columns={"Result": "label"})

# Save the updated dataset
df.to_csv("final_dataset_with_label.csv", index=False)

print("Column 'Result' has been renamed to 'label' and saved as 'final_dataset_with_label.csv'.")


Column 'Result' has been renamed to 'label' and saved as 'final_dataset_with_label.csv'.


In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("final_feature_dataset_cleaned.csv")



# Drop weak/noisy features (based on feature importance analysis)
noisy_features = [

    "has_escape",
    "has_popup",
    "css_imports",
    "has_eval",
    "has_onmouseover"
]

# Only drop those that exist in dataset
df = df.drop(columns=[f for f in noisy_features if f in df.columns])

# Save final dataset
df.to_csv("final_feature_dataset_selected.csv", index=False)
print(" Cleaned dataset saved as final_feature_dataset_selected.csv")

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv("final_feature_dataset_cleaned.csv")



# Split features and label
X = df.drop(columns=["label"])
y = df["label"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train RandomForest
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Feature importance
importances = model.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)

# Plot
plt.figure(figsize=(10,6))
feat_imp.plot(kind="bar")
plt.title("Feature Importance (RandomForest)")
plt.show()

# Check baseline accuracy
y_pred = model.predict(X_test)
print("Baseline Accuracy:", accuracy_score(y_test, y_pred))
